# **Modèle d'évaluation des risques de crédit à l'aide d'un algorithme "KNN"**

## 1. Installer des librairies nécessaires

In [1]:
pip install scikit-learn # Installer la librairie Scikit-learn

In [2]:
pip install gradio # Installer la librairie Gradio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 MB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.8/319.8 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.6/94.6 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.0/78.0 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 436.6/436.6 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.9/141.9 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 67.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.5/71.5 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/1

In [3]:
pip install colorama # Installer la librairie Colorama

In [4]:
# Importer les librairies nécessaires
import pandas as pd  # Pour la manipulation des données
from google.colab import files  # Pour télécharger des fichiers dans Google Colab
from sklearn.model_selection import train_test_split  # Pour diviser les données en ensembles d'entraînement et de test
from sklearn.neighbors import KNeighborsClassifier  # Pour utiliser le modèle de KNN
from sklearn.preprocessing import StandardScaler, OneHotEncoder  # Pour la mise à l'échelle et l'encodage des données
from sklearn.impute import SimpleImputer  # Pour gérer les valeurs manquantes
from sklearn.pipeline import Pipeline  # Pour créer un pipeline de prétraitement
from sklearn.compose import ColumnTransformer  # Pour appliquer différents transformateurs à différentes colonnes
from sklearn.metrics import accuracy_score, classification_report  # Pour évaluer le modèle
import gradio as gr # Pour créer des interfaces graphiques interactives Gradio
from colorama import Fore, Style  # Pour formatter l'affichage de messages dans la console

## 2. Chargement et affichage des données

In [5]:
# Télécharger le fichier de données dans Google Colab
uploaded = files.upload()

Saving credit_risk_dataset.csv to credit_risk_dataset.csv


In [6]:
# Lire les données depuis le fichier CSV
data = pd.read_csv('credit_risk_dataset.csv')

In [7]:
# Afficher les 5 premières lignes des données
data.head()

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,0.59,Y,3
1,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,N,2
2,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,N,3
3,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,0.53,N,2
4,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,0.55,Y,4


## 3. Préparation des données

In [8]:
# Prépare les données pour l'entraînement du modèle
def preparer_donnees(data):
    # Séparer les caractéristiques (X) de la variable cible (y)
    X = data[["person_age", "person_income", "person_emp_length", "loan_amnt", "loan_int_rate", "person_home_ownership", "loan_intent", "cb_person_default_on_file"]]
    y = data["loan_status"]

    # Définir les caractéristiques numériques et catégorielles
    numeric_features = ["person_age", "person_income", "person_emp_length", "loan_amnt", "loan_int_rate"]
    categorical_features = ["person_home_ownership", "loan_intent", "cb_person_default_on_file"]

    # Créer un pipeline pour les caractéristiques numériques
    numeric_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])

    # Créer un pipeline pour les caractéristiques catégorielles
    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ])

    # Combiner les pipelines numériques et catégorielles
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numeric_features),
            ('cat', categorical_transformer, categorical_features)
        ]
    )

    # Prétraiter les données
    X_processed = preprocessor.fit_transform(X)

    # Obtenir les noms des colonnes catégorielles après l'encodage
    categorical_columns = preprocessor.named_transformers_['cat'].named_steps['onehot'].get_feature_names_out(categorical_features)

    # Combiner les noms de toutes les colonnes
    all_columns = numeric_features + list(categorical_columns)

    # Diviser les données en ensembles d'entraînement et de test
    X_train, X_test, y_train, y_test = train_test_split(X_processed, y, test_size=0.2, random_state=42)

    # Retourne les données d'entraînement et de test, le préprocesseur et la liste des colonnes
    return X_train, X_test, y_train, y_test, preprocessor, all_columns

## 4. Entraînement du modèle

In [9]:
# Entraîne un modèle de KNN sur les données d'entraînement fournies
def entrainer_modele(X_train, y_train):
    # Crée une instance du modèle de KNN
    knn_classifier = KNeighborsClassifier(n_neighbors=5)

    # Entraîne le modèle sur les données d'entraînement
    knn_classifier.fit(X_train,y_train)

    # Retourne le modèle entraîné
    return knn_classifier

## 5. Évaluation du modèle

In [10]:
# Évalue les performances d'un modèle de classification sur les données de test
def evaluer_modele(knn_classifier, X_test, y_test):
    # Prédit les étiquettes de classe pour les données de test
    y_pred = knn_classifier.predict(X_test)

    # Calcule et affiche la précision du modèle
    accuracy = accuracy_score(y_test, y_pred)

    # Affiche un rapport de classification détaillé
    print(f"Précision du modèle: {accuracy:.2f}")
    print("\nRapport de Classification :")
    print(classification_report(y_test,y_pred))

## 6. Acquisition des données et validation des entrées

In [11]:
# Demande à l'utilisateur de saisir les détails du demandeur de prêt et prédit la probabilité de défaut de paiement
def predict_loan_default():
    print("\nEntrez les details du demandeur pour l'evaluation du risque de credit :")

    # Validation de l'âge
    while True:
        try:
            age = int(input("Age du demandeur :"))
            if 16 <= age <= 65:
                break
            else:
                print("Veuillez entrer un âge valide entre 16 et 65.")
        except ValueError:
            print("Veuillez entrer un âge valide (nombre entier).")

    # Validation du revenu
    while True:
        try:
            income = int(input("Revenu du demandeur :"))
            if 5000 <= income <= 1000000:
                break
            else:
                print("Veuillez entrer un revenu valide entre 5000 et 1000000.")
        except ValueError:
            print("Veuillez entrer un revenu valide (nombre entier).")

    # Validation de la durée de l'emploi
    while True:
        try:
            emp_length = int(input("Duree de l'emploi du demandeur (en annees) :"))
            if 1 <= emp_length <= 45:
                break
            else:
                print("Veuillez entrer une duree d'emploi valide entre 1 et 45 annees.")
        except ValueError:
            print("Veuillez entrer une durée d'emploi valide (nombre entier).")

    # Validation du montant du prêt
    while True:
        try:
            loan_amnt = int(input("Montant du pret demande :"))
            if 500 <= loan_amnt <= 99999:
                break
            else:
                print("Veuillez entrer un montant de prêt valide entre 500 et 99999.")
        except ValueError:
            print("Veuillez entrer un montant de prêt valide (nombre entier).")

    # Validation du taux d'intérêt
    while True:
        try:
            loan_int_rate = float(input("Taux d'interet du pret (ex: 5.01, 7.30):"))
            if 0.10 <= loan_int_rate <= 30.00:
                break
            else:
                print("Veuillez entrer un taux d'intérêt valide entre 0.10 et 30.00.")
        except ValueError:
            print("Veuillez entrer un taux d'intérêt valide (nombre décimal).")

    # Validation de la propriété du logement
    while True:
        home_ownership = input("Propriete du logement (RENT/MORTGAGE/OWN/OTHER):").upper()
        if home_ownership in ["RENT", "MORTGAGE", "OWN", "OTHER"]:
            break
        else:
            print("Veuillez entrer une valeur valide pour la propriété du logement (RENT, MORTGAGE, OWN, ou OTHER).")

    # Validation de l'objet du prêt
    while True:
        loan_intent = input("Objet du pret (DEBTCONSOLIDATION/EDUCATION/HOMEIMPROVEMENT/MEDICAL/PERSONAL/VENTURE):").upper()
        if loan_intent in ["DEBTCONSOLIDATION", "EDUCATION", "HOMEIMPROVEMENT", "MEDICAL", "PERSONAL", "VENTURE"]:
            break
        else:
            print("Veuillez entrer un objet de prêt valide (DEBTCONSOLIDATION, EDUCATION, HOMEIMPROVEMENT, MEDICAL, PERSONAL, ou VENTURE).")

    # Validation de l'historique de défaut de paiement
    while True:
        default_history = input("Le demandeur a-t-il deja fait defaut de paiement (Y/N) ? :").upper()
        if default_history in ["Y", "N"]:
            break
        else:
            print("Veuillez entrer une valeur valide pour l'historique de défaut de paiement (Y ou N).")

    return age, income, emp_length, loan_amnt, loan_int_rate, home_ownership, loan_intent, default_history

## 7. Préparation des données pour la prédiction

In [12]:
# Prépare les données d'entrée pour la prédiction en créant un DataFrame Pandas et en appliquant le prétraitement nécessaire
def preparer_donnees_pour_prediction(age, income, emp_length, loan_amnt, loan_int_rate, home_ownership, loan_intent, default_history, preprocessor, all_columns):
    # Crée un DataFrame Pandas avec les données d'entrée
    input_data = pd.DataFrame({
        "person_age": [age],
        "person_income": [income],
        "person_emp_length": [emp_length],
        "loan_amnt": [loan_amnt],
        "loan_int_rate": [loan_int_rate],
        "person_home_ownership": [home_ownership],
        "loan_intent": [loan_intent],
        "cb_person_default_on_file": [default_history]
    })

    # Applique le prétraitement aux données d'entrée en utilisant le preprocessor
    input_processed = preprocessor.transform(input_data)

    # Retourne les données d'entrée prétraitées
    return input_processed

## 8. Prédiction et affichage du résultat

In [13]:
def predire_et_afficher_resultat(knn_classifier, input_data):
    # Utilise le modèle pour prédire la classe (0 ou 1) pour les données d'entrée
    prediction = knn_classifier.predict(input_data)[0]

    # Affiche un message à l'utilisateur en fonction de la prédiction
    if prediction == 1:
        print(Fore.BLUE + Style.BRIGHT + "D'après les informations fournies, il est prédit que le demandeur est PLUS susceptible de faire DEFAUT sur le prêt." + Style.RESET_ALL)
    else:
        print(Fore.BLUE + Style.BRIGHT + "D'après les informations fournies, il est prédit que le demandeur est MOINS susceptible de faire DEFAUT sur le prêt." + Style.RESET_ALL)

## 9. Appeler la fonction pour prédire le défaut de prêt

In [14]:
# Fonction principale pour charger les données, entraîner le modèle, évaluer le modèle et prédire le risque de crédit pour de nouveaux demandeurs
def main():
    # Charger les données (assurez-vous que 'data' est défini correctement)
    data = pd.read_csv("credit_risk_dataset.csv")  # Assurez-vous que le chemin du fichier est correct

    # Préparer les données
    X_train, X_test, y_train, y_test, preprocessor, all_columns = preparer_donnees(data)

    # Entraîner le modèle
    knn_classifier = entrainer_modele(X_train, y_train)  # Entraîner le modèle KNN

    # Évaluer le modèle
    evaluer_modele(knn_classifier, X_test, y_test)  # Évaluer le modèle KNN

    # Boucle principale pour les prédictions
    while True:
        # Collecter les données de l'utilisateur
        age, income, emp_length, loan_amnt, loan_int_rate, home_ownership, loan_intent, default_history = predict_loan_default()

        # Préparer les données pour la prédiction
        input_data = preparer_donnees_pour_prediction(age, income, emp_length, loan_amnt, loan_int_rate, home_ownership, loan_intent, default_history, preprocessor, all_columns)

        # Prédire et afficher le résultat
        predire_et_afficher_resultat(knn_classifier, input_data)  # Utiliser le modèle KNN

        # Demander à l'utilisateur s'il souhaite faire une autre prédiction
        continuer = input("\nVoulez-vous faire une autre prédiction ? (O/N) : ").upper()
        if continuer != 'O':
            break

    print(Fore.BLUE + Style.BRIGHT + "Merci d'avoir utilisé notre système d'IA 'CleVI' de prédiction de risque de crédit!" + Style.RESET_ALL)

if __name__ == "__main__":
    main()

Précision du modèle: 0.84

Rapport de Classification :
              precision    recall  f1-score   support

           0       0.86      0.95      0.90      5072
           1       0.72      0.48      0.58      1445

    accuracy                           0.84      6517
   macro avg       0.79      0.71      0.74      6517
weighted avg       0.83      0.84      0.83      6517


Entrez les details du demandeur pour l'evaluation du risque de credit :
Age du demandeur :33
Revenu du demandeur :33333
Duree de l'emploi du demandeur (en annees) :3
Montant du pret demande :33333
Taux d'interet du pret (ex: 5.01, 7.30):8
Propriete du logement (RENT/MORTGAGE/OWN/OTHER):OWN
Objet du pret (DEBTCONSOLIDATION/EDUCATION/HOMEIMPROVEMENT/MEDICAL/PERSONAL/VENTURE):EDUCATION
Le demandeur a-t-il deja fait defaut de paiement (Y/N) ? :N
D'après les informations fournies, il est prédit que le demandeur est MOINS susceptible de faire DEFAUT sur le prêt.

Voulez-vous faire une autre prédiction ? (O/N) : N
Me

## 10. Interface Gradio pour la prédiction de défaut de prêt

In [15]:
# Fonction pour la prédiction avec Gradio
def gradio_predict_loan_default(age, income, emp_length, loan_amnt, loan_int_rate, home_ownership, loan_intent, default_history):
    # Créer un DataFrame avec les données d'entrée
    input_data = pd.DataFrame({
        "person_age": [age],
        "person_income": [income],
        "person_emp_length": [emp_length],
        "loan_amnt": [loan_amnt],
        "loan_int_rate": [loan_int_rate],
        "person_home_ownership": [home_ownership],
        "loan_intent": [loan_intent],
        "cb_person_default_on_file": [default_history]
    })

    # Prétraiter les données d'entrée
    input_processed = preprocessor.transform(input_data)

    # Faire la prédiction avec le modèle KNN
    prediction = knn_classifier.predict(input_processed)[0]

    # Retourner le résultat
    if prediction == 1:
        return "D'après les informations fournies, il est prédit que le demandeur est PLUS susceptible de faire DÉFAUT sur le prêt."
    else:
        return "D'après les informations fournies, il est prédit que le demandeur est MOINS susceptible de faire DÉFAUT sur le prêt."

# Fonction principale
def main():
    global preprocessor, knn_classifier  # Rendre ces variables globales pour les utiliser dans gradio_predict_loan_default

    # Charger les données
    data = pd.read_csv("credit_risk_dataset.csv")  # Assurez-vous que le chemin du fichier est correct

    # Préparer les données
    X_train, X_test, y_train, y_test, preprocessor, all_columns = preparer_donnees(data)

    # Entraîner le modèle KNN
    knn_classifier = entrainer_modele(X_train, y_train)

    # Évaluer le modèle
    evaluer_modele(knn_classifier, X_test, y_test)

    # Créer l'interface Gradio
    iface = gr.Interface(
        fn=gradio_predict_loan_default,
        inputs=[
            gr.Number(label="Age du demandeur", minimum=16, maximum=65),
            gr.Number(label="Revenu du demandeur", minimum=5000, maximum=1000000),
            gr.Number(label="Durée de l'emploi du demandeur (en années)", minimum=1, maximum=45),
            gr.Number(label="Montant du prêt demande", minimum=500, maximum=99999),
            gr.Number(label="Taux d'intérêt du prêt (ex: 5.01, 7.30)", minimum=0.10, maximum=30.00),
            gr.Dropdown(choices=['RENT', 'MORTGAGE', 'OWN', 'OTHER'], label="Propriété du logement"),
            gr.Dropdown(choices=['DEBTCONSOLIDATION', 'EDUCATION', 'HOMEIMPROVEMENT', 'MEDICAL', 'PERSONAL', 'VENTURE'], label="Objet du prêt"),
            gr.Dropdown(choices=['Y', 'N'], label="Le demandeur a-t-il déjà fait défaut de paiement ?")
        ],
        outputs="text",
        title="Système d'IA 'CleVI' : Évaluation du Risque de Crédit (Interface Gradio)"
    )

    # Lancer l'interface Gradio
    iface.launch()

if __name__ == "__main__":
    main()

Précision du modèle: 0.84

Rapport de Classification :
              precision    recall  f1-score   support

           0       0.86      0.95      0.90      5072
           1       0.72      0.48      0.58      1445

    accuracy                           0.84      6517
   macro avg       0.79      0.71      0.74      6517
weighted avg       0.83      0.84      0.83      6517

Setting queue=True in a Colab notebook requires sharing enabled. Setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://998e39c8394e3dcad7.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
predict_loan_default()


Enter applicant details for credit risk assessment:
Applicant's age:120
Applicant's income:20
Applicant's employment length (in years):100
Loan interest rate:100
Loan amount requested:20
Home ownership (RENT/MORTGAGE/OWN/OTHER):NIMPORTE QUOI
Loan intent (DEBTCONSOLIDATION/EDUCATION/HOMEIMPROVEMENT/MEDICAL/PERSONAL/VENTURE):GRAND IDEA
Has the applicant defaulted before (Y/N):100

Based on the information provided, the applicant is predicted to be less likely to default on the loan


/usr/local/lib/python3.10/dist-packages/sklearn/base.py:486: UserWarning: X has feature names, but KNeighborsClassifier was fitted without feature names
  warnings.warn(
